# Next Steps — RQ2 extension and new RQ3 (step by step)
# ขั้นตอนต่อไป — ขยาย RQ2 และ RQ3 แบบใหม่ (ทีละขั้น)

**Goal / เป้าหมาย**
- Make the 62 GB CSV small and fast to use (convert once to Parquet). / ทำให้ไฟล์ 62 GB เล็กและใช้งานเร็ว (แปลงเป็น Parquet ครั้งเดียว)
- **New RQ3:** where and when is outage risk higher? (season, technology, region, time of day) / **RQ3 ใหม่:** ที่ไหน ช่วงไหน เสี่ยง outage มากกว่า
- **RQ2 extension:** does neighbor impact depend on outage duration, same site, technology, time of day? / **ขยาย RQ2:** ผลกระทบต่อ neighbor ขึ้นกับระยะเวลา outage, ไซต์เดียวกัน, เทคโนโลยี, ช่วงเวลา หรือไม่

**Run the cells in order, one step at a time. Check the "What you should see" box after each step before moving on.**
**รันทีละ cell ตามลำดับ และเช็กกล่อง "ผลที่ควรเห็น" ทุกขั้นก่อนไปขั้นต่อไป**

| Step | What | Time (approx.) |
|---|---|---|
| 0 | Preparation and paths / เตรียมเครื่องและ path | 2 min |
| 1 | Convert CSV → Parquet (one time) / แปลงไฟล์ (ครั้งเดียว) | 15–45 min |
| 2 | Check the Parquet file / ตรวจไฟล์ | 1 min |
| 3 | Build the outage table / สร้างตาราง outage | 1 min |
| 4 | New RQ3: outage risk model / โมเดลความเสี่ยง | 1 min |
| 5 | RQ2 extension: neighbor impact by group / ผลต่อ neighbor แยกกลุ่ม | 5–15 min |
| 6 | Summary for the report / สรุปไปใส่รีพอร์ต | 1 min |

## Step 0 — Preparation / เตรียมความพร้อม

**Before you run / ก่อนรัน**
1. Close other big programs (Chrome tabs, games). / ปิดโปรแกรมใหญ่อื่น ๆ
2. Make sure drive D: has **at least 10 GB free** (Parquet file ≈ 3–6 GB + temp files). / ไดรฟ์ D: ต้องว่าง **อย่างน้อย 10 GB**
3. Keep the laptop plugged in; Step 1 reads the whole 62 GB once. / เสียบสายชาร์จไว้ ขั้นที่ 1 อ่านไฟล์ 62 GB ทั้งไฟล์หนึ่งรอบ
4. Check the two paths below match your computer. / ตรวจ path สองบรรทัดด้านล่างให้ตรงกับเครื่อง
5. If your computer has **8 GB RAM or less**, change `MEMORY_LIMIT` to `"2GB"`. / ถ้าเครื่องมี RAM ≤ 8 GB ให้เปลี่ยน `MEMORY_LIMIT` เป็น `"2GB"`

**What you should see / ผลที่ควรเห็น:** both files `True`, free space ≥ 10 GB, library versions printed, no error.

In [ ]:
# Run once if a library is missing (remove the # and run), then restart the kernel
# %pip install duckdb pyarrow statsmodels scikit-learn scipy matplotlib pandas

In [ ]:
# ============================================================
# Step 0 — configuration
# ============================================================
from pathlib import Path
import shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb, statsmodels, scipy
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import binomtest, spearmanr

# ---- EDIT THESE PATHS IF NEEDED / แก้ path ตรงนี้ถ้าจำเป็น ----
DATASET_1_FILE = Path(r"D:\Dataset\Uganda_Cell_Telecom_Dataset\Dataset_1_Cell_Information_V3.csv")
DATASET_2_CSV  = Path(r"D:\Dataset\Uganda_Cell_Telecom_Dataset\Dataset_2_Marge.csv")
WORK_DIR       = Path(r"D:\Dataset\Uganda_Cell_Telecom_Dataset\next_steps")      # everything new goes here
MEMORY_LIMIT   = "4GB"                           # use "2GB" if RAM <= 8 GB
THREADS        = 4
RANDOM_STATE   = 42
# -----------------------------------------------------------------

PARQUET_FILE = WORK_DIR / "dataset2.parquet"
OUT_DIR = WORK_DIR / "outputs"
TMP_DIR = WORK_DIR / "duckdb_tmp"
for d in (WORK_DIR, OUT_DIR, TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

def sql_path(p):
    # DuckDB wants forward slashes and escaped quotes
    return str(p).replace("\\", "/").replace("'", "''")

con = duckdb.connect()
con.execute(f"SET temp_directory = '{sql_path(TMP_DIR)}'")
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute("SET preserve_insertion_order = false")

free_gb = shutil.disk_usage(WORK_DIR).free / 1e9
print("Dataset 1 exists:", DATASET_1_FILE.is_file())
print("Dataset 2 CSV exists:", DATASET_2_CSV.is_file(),
      f"({DATASET_2_CSV.stat().st_size / 1e9:.1f} GB)" if DATASET_2_CSV.is_file() else "")
print(f"Free space on the work drive: {free_gb:.1f} GB  (need >= 10 GB)")
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| statsmodels", statsmodels.__version__)
if free_gb < 10 and not PARQUET_FILE.exists():
    print("WARNING: less than 10 GB free. Free some space before Step 1.")

## Step 1 — Convert the 62 GB CSV to Parquet (one time only) / แปลง CSV 62 GB เป็น Parquet (ครั้งเดียว)

**What it does / ทำอะไร:** DuckDB reads the CSV in a stream (not all into RAM) and writes a compressed Parquet file with proper data types. Parquet is usually 5–15× smaller and 10–100× faster to query.
DuckDB อ่าน CSV แบบไหลทีละส่วน (ไม่โหลดเข้า RAM ทั้งหมด) แล้วเขียนเป็นไฟล์ Parquet ที่บีบอัดและมีชนิดข้อมูลถูกต้อง Parquet มักเล็กกว่า 5–15 เท่า และ query เร็วกว่า 10–100 เท่า

**Notes / หมายเหตุ**
- Timestamps that are written as date only (`2024-01-01`) are read correctly as midnight. / Timestamp ที่เขียนแค่วันที่จะถูกอ่านเป็นเที่ยงคืนให้อัตโนมัติ
- It writes to a temporary name first, so a crash never leaves a half-finished file. / เขียนเป็นชื่อชั่วคราวก่อน ถ้าเครื่องค้างกลางทางจะไม่มีไฟล์ครึ่ง ๆ กลาง ๆ
- If the file already exists, this step is skipped. / ถ้ามีไฟล์แล้ว ขั้นนี้จะข้ามให้เอง

**What you should see / ผลที่ควรเห็น:** `Done in XX min` and a Parquet size of roughly **3–6 GB**. Takes about 15–45 min (depends on disk speed).

In [ ]:
# ============================================================
# Step 1 — CSV -> Parquet (one pass over the 62 GB file)
# ============================================================
NUM_FLOAT = ["Traffic", "Connected_Users", "Throughput", "Latency", "Packet_Loss",
             "Load", "Capacity", "Alarm_Count"]

if PARQUET_FILE.exists():
    print(f"Already exists, skipping: {PARQUET_FILE} ({PARQUET_FILE.stat().st_size / 1e9:.2f} GB)")
else:
    tmp_file = PARQUET_FILE.with_suffix(".parquet.tmp")
    if tmp_file.exists():
        tmp_file.unlink()
    float_cols = ",\n        ".join(f"TRY_CAST({c} AS FLOAT) AS {c}" for c in NUM_FLOAT)
    query = f'''
    COPY (
      SELECT
        TRY_CAST(Timestamp AS TIMESTAMP)      AS Timestamp,
        Site_ID, Cell_ID, Record_Type,
        {float_cols},
        TRY_CAST(Previous_Outage_Count AS INTEGER) AS Previous_Outage_Count,
        TRY_CAST(Cell_Down_Time AS TIMESTAMP) AS Cell_Down_Time,
        TRY_CAST(Cell_Up_Time AS TIMESTAMP)   AS Cell_Up_Time
      FROM read_csv('{sql_path(DATASET_2_CSV)}', header = true, all_varchar = true)
    ) TO '{sql_path(tmp_file)}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)
    '''
    t0 = time.time()
    print("Converting... (this is the long step; do not close Jupyter)")
    con.execute(query)
    tmp_file.rename(PARQUET_FILE)
    print(f"Done in {(time.time() - t0) / 60:.1f} min -> {PARQUET_FILE} "
          f"({PARQUET_FILE.stat().st_size / 1e9:.2f} GB)")

PQ = f"read_parquet('{sql_path(PARQUET_FILE)}')"   # used by all later steps

## Step 2 — Check the Parquet file / ตรวจไฟล์ Parquet

**What it does / ทำอะไร:** counts rows by record type and checks for unreadable timestamps, so you know nothing was lost.
นับจำนวนแถวแต่ละประเภท และเช็ก timestamp ที่อ่านไม่ได้ เพื่อยืนยันว่าข้อมูลไม่หาย

**What you should see (full data) / ผลที่ควรเห็น (ข้อมูลเต็ม):**

| Record_Type | Rows |
|---|---|
| KPI | 133,908,458 |
| OUTAGE | 25,341 |
| RECOVERY | 25,333 |

and `Bad_Timestamp = 0`. If the numbers match, the CSV is no longer needed for this notebook — you may move it to an external drive (keep a backup!).
และ `Bad_Timestamp = 0` ถ้าตัวเลขตรง notebook นี้ไม่ต้องใช้ CSV อีกแล้ว ย้ายไปเก็บในฮาร์ดดิสก์ภายนอกได้ (เก็บสำรองไว้ด้วย)

In [ ]:
# ============================================================
# Step 2 — sanity check
# ============================================================
check = con.execute(f'''
    SELECT Record_Type,
           COUNT(*)                         AS Rows,
           SUM(Timestamp IS NULL)::BIGINT   AS Bad_Timestamp,
           MIN(Timestamp)                   AS First,
           MAX(Timestamp)                   AS Last,
           COUNT(DISTINCT Cell_ID)          AS Cells
    FROM {PQ} GROUP BY Record_Type ORDER BY Record_Type
''').df()
display(check)
print("Total rows:", f"{check['Rows'].sum():,}")
check.to_csv(OUT_DIR / "step2_parquet_check.csv", index=False)

## Step 3 — Build the outage table / สร้างตาราง outage

**What it does / ทำอะไร:** takes only the ~50,000 OUTAGE/RECOVERY rows (seconds, thanks to Parquet), matches each outage to its recovery to get the duration, and adds technology, region, district, season and 3-hour time block from Dataset 1.
ดึงเฉพาะแถว OUTAGE/RECOVERY (~50,000 แถว ใช้เวลาไม่กี่วินาที) จับคู่ outage กับ recovery เพื่อหาระยะเวลา แล้วเติมเทคโนโลยี ภูมิภาค อำเภอ ฤดู และช่วงเวลา 3 ชั่วโมง จาก Dataset 1

**What you should see (full data) / ผลที่ควรเห็น:** about **25,333 matched outages**, duration from **0.5 to 4.0 h**, mean ≈ **2.24 h** (same as your EDA). File `outages_clean.parquet` saved.

In [ ]:
# ============================================================
# Step 3 — outage table with duration and context
# ============================================================
df1 = pd.read_csv(DATASET_1_FILE, low_memory=False)
df1["Cell_ID"] = df1["Cell_ID"].astype(str)

ev = con.execute(f'''
    SELECT Cell_ID, Record_Type, Timestamp, Cell_Down_Time, Cell_Up_Time
    FROM {PQ} WHERE Record_Type IN ('OUTAGE', 'RECOVERY')
''').df()
out = (ev[ev.Record_Type == "OUTAGE"].dropna(subset=["Cell_Down_Time"])
       [["Cell_ID", "Cell_Down_Time"]].sort_values("Cell_Down_Time"))
rec = (ev[ev.Record_Type == "RECOVERY"].dropna(subset=["Cell_Up_Time"])
       [["Cell_ID", "Cell_Up_Time"]].sort_values("Cell_Up_Time"))

# next recovery of the same cell at or after the outage start
outages = pd.merge_asof(out, rec, left_on="Cell_Down_Time", right_on="Cell_Up_Time",
                        by="Cell_ID", direction="forward")
outages = outages.dropna(subset=["Cell_Up_Time"])
outages = outages.drop_duplicates(subset=["Cell_ID", "Cell_Up_Time"], keep="first")  # one recovery per outage
outages["Duration_h"] = (outages.Cell_Up_Time - outages.Cell_Down_Time).dt.total_seconds() / 3600

def project_season(m):          # same definition as the main notebook
    if m in (2, 3, 4, 5):  return "Summer"
    if m in (6, 7, 8, 9):  return "Rainy Season"
    return "Winter"

outages = outages.merge(df1[["Cell_ID", "Site_ID", "Technology", "Region", "District"]],
                        on="Cell_ID", how="left")
t = outages.Cell_Down_Time
outages["Month"] = t.dt.to_period("M").astype(str)
outages["Season"] = t.dt.month.map(project_season)
outages["Block"] = (t.dt.hour // 3 * 3).map(lambda h: f"{h:02d}-{h + 3:02d}h")
outages["Duration_Group"] = pd.cut(outages.Duration_h, [0, 1, 2, 3, 4.01, np.inf],
                                   labels=["<1 h", "1-2 h", "2-3 h", "3-4 h", ">4 h"], right=False)

print(f"Outages: {len(out):,} | matched to a recovery: {len(outages):,}")
print("Missing technology/region after join:", int(outages.Technology.isna().sum()))
display(outages.Duration_h.describe().to_frame("Duration (hours)").round(3))
display(outages.groupby("Technology").size().to_frame("Outages"))
outages.to_parquet(WORK_DIR / "outages_clean.parquet", index=False)

## Step 4 — New RQ3: where and when is outage risk higher? / RQ3 ใหม่: ที่ไหน ช่วงไหน เสี่ยงมากกว่า

**What it does / ทำอะไร**
1. Counts outages for every **Region × Technology × Month × 3-hour block** (including zero counts). / นับ outage ทุกกลุ่ม ภูมิภาค × เทคโนโลยี × เดือน × ช่วง 3 ชม. (รวมกลุ่มที่เป็นศูนย์)
2. Fits a **Poisson regression** with exposure = number of cells × days (so big regions are not unfairly "riskier"). / ใช้ **Poisson regression** โดยหารด้วยจำนวนเซลล์ × จำนวนวัน (ภูมิภาคใหญ่จะไม่ดูเสี่ยงเกินจริง)
3. Reports the **Incidence Rate Ratio (IRR)**: IRR 1.10 = 10% higher outage rate than the reference group. / รายงาน **IRR**: 1.10 = อัตรา outage สูงกว่ากลุ่มอ้างอิง 10%
4. Forecast check: train on the first half of the months, predict monthly outages per region in the second half, compare with a simple average. / ทดสอบพยากรณ์: เทรนครึ่งแรก ทำนายครึ่งหลัง เทียบกับค่าเฉลี่ยธรรมดา

**What you should see / ผลที่ควรเห็น:** a table and a chart of IRRs with 95% CI. From your EDA, expect **Rainy Season > 1**, and **2G/3G/4G > 1 vs 5G** (around 1.1–1.2). Effects are real but modest — say so honestly. The forecast table should show the model with a lower error than the flat average.
ตาราง + กราฟ IRR พร้อม 95% CI จาก EDA คาดว่า **Rainy Season > 1** และ **2G/3G/4G > 1 เมื่อเทียบกับ 5G** (ประมาณ 1.1–1.2) ผลมีจริงแต่ไม่ใหญ่ ให้เขียนตามจริง ตารางพยากรณ์ควรให้ error ของโมเดลต่ำกว่าค่าเฉลี่ยธรรมดา

In [ ]:
# ============================================================
# Step 4A — Poisson model of outage rate (IRR)
# ============================================================
cells_rt = df1.groupby(["Region", "Technology"]).size().rename("N_Cells").reset_index()
months = pd.period_range(outages.Cell_Down_Time.min(), outages.Cell_Down_Time.max(), freq="M")
blocks = [f"{h:02d}-{h + 3:02d}h" for h in range(0, 24, 3)]
grid = (cells_rt.merge(pd.DataFrame({"Month": months.astype(str)}), how="cross")
                .merge(pd.DataFrame({"Block": blocks}), how="cross"))
counts = outages.groupby(["Region", "Technology", "Month", "Block"]).size().rename("Outages").reset_index()
grid = grid.merge(counts, how="left").fillna({"Outages": 0})
grid["Days"] = pd.PeriodIndex(grid.Month, freq="M").days_in_month
grid["Season"] = pd.PeriodIndex(grid.Month, freq="M").month.map(project_season)
grid["log_exposure"] = np.log(grid.N_Cells * grid.Days)

def ref(col, wanted):           # reference level if present, else the first level
    levels = sorted(grid[col].unique())
    return wanted if wanted in levels else levels[0]

formula = (f"Outages ~ C(Season, Treatment('{ref('Season', 'Winter')}'))"
           f" + C(Technology, Treatment('{ref('Technology', '5G')}'))"
           f" + C(Region, Treatment('{ref('Region', 'Central')}'))"
           f" + C(Block, Treatment('{ref('Block', '00-03h')}'))")
glm = smf.glm(formula, data=grid, family=sm.families.Poisson(), offset=grid.log_exposure).fit(scale="X2")
print(f"Rows in model: {len(grid):,} | total outages: {int(grid.Outages.sum()):,}")
print(f"Dispersion (Pearson chi2 / df): {glm.pearson_chi2 / glm.df_resid:.2f}  (about 1 = Poisson fits well; "
      "CIs below are already adjusted for it)")

ci = glm.conf_int()
irr = pd.DataFrame({"IRR": np.exp(glm.params), "CI_low": np.exp(ci[0]), "CI_high": np.exp(ci[1]),
                    "p_value": glm.pvalues}).drop(index="Intercept")
irr.index = (irr.index.str.replace(r"C\((\w+), Treatment\('[^']*'\)\)\[T\.", r"\1: ", regex=True)
                      .str.rstrip("]"))
display(irr.round(3))
irr.to_csv(OUT_DIR / "step4_rq3_irr.csv")

fig, ax = plt.subplots(figsize=(8, 0.38 * len(irr) + 1))
y = np.arange(len(irr))[::-1]
ax.errorbar(irr.IRR, y, xerr=[irr.IRR - irr.CI_low, irr.CI_high - irr.IRR], fmt="o", color="#2a78d6", capsize=3)
ax.axvline(1, color="gray", ls="--")
ax.set_yticks(y); ax.set_yticklabels(irr.index)
ax.set_xlabel("Incidence Rate Ratio (95% CI)  —  1 = same risk as reference group")
ax.set_title("RQ3: which groups have a higher outage rate?", loc="left")
plt.tight_layout(); plt.savefig(OUT_DIR / "step4_rq3_irr.png", dpi=200); plt.show()

In [ ]:
# ============================================================
# Step 4B — forecast check: first half of months -> second half
# ============================================================
all_months = sorted(grid.Month.unique())
cut = all_months[len(all_months) // 2]
train, test = grid[grid.Month < cut].copy(), grid[grid.Month >= cut].copy()
formula_fc = formula
if not set(test.Season) <= set(train.Season):     # only happens with less than a year of data
    formula_fc = "Outages ~ " + formula.split(" + ", 1)[1]
    print("Note: some seasons are missing from the training months, so Season is left out of the forecast model.")
glm_tr = smf.glm(formula_fc, data=train, family=sm.families.Poisson(), offset=train.log_exposure).fit()
test["Model"] = glm_tr.predict(test, offset=test.log_exposure)

# baseline: the region's average outages per cell-day in the training period
exposure = (train.N_Cells * train.Days).groupby(train.Region).sum() / len(blocks)
rate = train.groupby("Region").Outages.sum() / exposure
test["Average"] = test.Region.map(rate) * test.N_Cells * test.Days / len(blocks)

monthly = test.groupby(["Region", "Month"])[["Outages", "Model", "Average"]].sum()
res = pd.DataFrame({
    "MAE (outages per region-month)": [(monthly.Model - monthly.Outages).abs().mean(),
                                       (monthly.Average - monthly.Outages).abs().mean()],
    "MAPE %": [((monthly.Model - monthly.Outages).abs() / monthly.Outages.clip(lower=1)).mean() * 100,
               ((monthly.Average - monthly.Outages).abs() / monthly.Outages.clip(lower=1)).mean() * 100],
}, index=["Poisson model", "Flat average (baseline)"])
print(f"Train months: {all_months[0]} .. {all_months[len(all_months)//2 - 1]} | test months: {cut} .. {all_months[-1]}")
display(res.round(2))
res.to_csv(OUT_DIR / "step4_rq3_forecast_check.csv")

tot = monthly.groupby("Month").sum()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tot.index, tot.Outages, "o-", color="black", label="Actual")
ax.plot(tot.index, tot.Model, "s--", color="#2a78d6", label="Poisson model")
ax.plot(tot.index, tot.Average, ":", color="#eb6834", label="Flat average")
ax.set_ylabel("Outages per month (all regions)"); ax.legend(frameon=False)
ax.set_title("RQ3 forecast check on the later months", loc="left")
plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(OUT_DIR / "step4_rq3_forecast.png", dpi=200); plt.show()

## Step 5 — RQ2 extension: what makes neighbor impact bigger? / ขยาย RQ2: อะไรทำให้ neighbor โดนหนักขึ้น

**What it does / ทำอะไร**
1. Randomly picks **2,000 outages** (your first analysis used 300 → more power). / สุ่ม **2,000 outages** (ครั้งแรกใช้ 300 → ผลแม่นขึ้น)
2. Pulls only the neighbor KPI rows needed (12 h around each outage, and the same hours 7 days earlier) straight from Parquet. / ดึงเฉพาะแถว KPI ของ neighbor ที่ต้องใช้ (±6 ชม. รอบ outage และช่วงเดียวกันเมื่อ 7 วันก่อน) จาก Parquet โดยตรง
3. Computes the same **difference-in-differences (DiD)** as Section 6.2E. / คำนวณ **DiD** แบบเดียวกับ 6.2E
4. Splits the result by **same site vs other site, outage duration, technology of the failed cell, time of day**, with a sign test for each group. / แยกผลตาม **ไซต์เดียวกัน/ต่างไซต์, ระยะเวลา outage, เทคโนโลยี, ช่วงเวลา** และทดสอบ sign test ทุกกลุ่ม

**What you should see / ผลที่ควรเห็น:** about 20,000 outage–neighbor pairs. Share with an increase in Load / Connected_Users above 50% in most groups, probably **higher for same-site neighbors** (your first 300-outage run showed this), and probably **higher for longer outages** — KPIs are recorded every 3 h, so a 30-minute outage can end before the next KPI reading and leave no trace. That dose-response pattern is a good, explainable finding for the report.
ประมาณ 20,000 คู่ สัดส่วนที่ Load / Connected_Users เพิ่มขึ้นเกิน 50% ในเกือบทุกกลุ่ม น่าจะ **สูงกว่าในเซลล์ไซต์เดียวกัน** (ตามผลรอบแรก 300 outages) และน่าจะ **สูงขึ้นเมื่อ outage นานขึ้น** เพราะ KPI บันทึกทุก 3 ชม. outage 30 นาทีอาจจบก่อนการวัดครั้งถัดไปจึงไม่เห็นผล รูปแบบ "ยิ่งนานยิ่งกระทบ" นี้เป็นข้อค้นพบที่ดีและอธิบายได้ในรีพอร์ต

In [ ]:
# ============================================================
# Step 5A — sample outages and pull neighbor KPI windows from Parquet
# ============================================================
N_OUTAGES = 2000
WIN = pd.Timedelta(hours=6)
LAG = pd.Timedelta(days=7)
KPIS = ["Traffic", "Connected_Users", "Throughput", "Latency", "Packet_Loss", "Load"]

edges = df1[["Cell_ID", "Site_ID", "Neighbor_Cell_ID"]].copy()
edges["Neighbor_Cell_ID"] = edges.Neighbor_Cell_ID.fillna("").astype(str).str.split(",")
edges = edges.explode("Neighbor_Cell_ID")
edges["Neighbor_Cell_ID"] = edges.Neighbor_Cell_ID.str.strip()
edges = edges[edges.Neighbor_Cell_ID != ""]
CELL_SITE = df1.set_index("Cell_ID").Site_ID.astype(str).to_dict()
CELL_TECH = df1.set_index("Cell_ID").Technology.to_dict()

t_min, t_max = outages.Cell_Down_Time.min(), outages.Cell_Down_Time.max()
eligible = outages[(outages.Cell_Down_Time >= t_min + LAG + pd.Timedelta(days=1)) &
                   (outages.Cell_Down_Time <= t_max - pd.Timedelta(days=1))]
ev_s = eligible.sample(min(N_OUTAGES, len(eligible)), random_state=RANDOM_STATE).reset_index(drop=True)
ev_s["Event"] = np.arange(len(ev_s))

pairs = ev_s.merge(edges[["Cell_ID", "Neighbor_Cell_ID"]], on="Cell_ID")
pairs["Same_Site"] = pairs.Neighbor_Cell_ID.map(CELL_SITE) == pairs.Cell_ID.map(CELL_SITE)
print(f"Sampled outages: {len(ev_s):,} | outage-neighbor pairs: {len(pairs):,}")

# one time window per (neighbor, outage) for "now" and one for "7 days earlier"
win = pd.concat([
    pd.DataFrame({"Cell_ID": pairs.Neighbor_Cell_ID, "t0": pairs.Cell_Down_Time - WIN, "t1": pairs.Cell_Down_Time + WIN}),
    pd.DataFrame({"Cell_ID": pairs.Neighbor_Cell_ID, "t0": pairs.Cell_Down_Time - LAG - WIN, "t1": pairs.Cell_Down_Time - LAG + WIN}),
]).drop_duplicates()
con.register("win", win)
t0 = time.time()
kpi_win = con.execute(f'''
    SELECT DISTINCT k.Cell_ID, k.Timestamp, {", ".join("k." + c for c in KPIS)}
    FROM {PQ} AS k
    JOIN win AS w ON k.Cell_ID = w.Cell_ID AND k.Timestamp BETWEEN w.t0 AND w.t1
    WHERE k.Record_Type = 'KPI'
''').df()
con.unregister("win")
print(f"KPI rows pulled: {len(kpi_win):,} in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ============================================================
# Step 5B — difference-in-differences per outage-neighbor pair
# ============================================================
kpi_win = kpi_win.sort_values(["Cell_ID", "Timestamp"])
BY_CELL = {c: (g.Timestamp.values, g[KPIS].to_numpy(float)) for c, g in kpi_win.groupby("Cell_ID")}

def med(cell, a, b, right_closed=False):
    # median KPI values of `cell` in [a, b)  (or (a, b] if right_closed)
    if cell not in BY_CELL:
        return None
    ts, vals = BY_CELL[cell]
    side_a, side_b = ("right", "right") if right_closed else ("left", "left")
    i, j = np.searchsorted(ts, np.datetime64(a), side_a), np.searchsorted(ts, np.datetime64(b), side_b)
    return np.nanmedian(vals[i:j], axis=0) if j > i else None

rows = []
for p in pairs.itertuples(index=False):
    e, n = p.Cell_Down_Time, p.Neighbor_Cell_ID
    b, a = med(n, e - WIN, e), med(n, e, e + WIN, True)
    cb, ca = med(n, e - LAG - WIN, e - LAG), med(n, e - LAG, e - LAG + WIN, True)
    if any(x is None for x in (b, a, cb, ca)):
        continue
    rows.append([p.Event, p.Cell_ID, n, p.Same_Site, p.Technology, p.Duration_h,
                 str(p.Duration_Group), p.Block, p.Season, *((a - b) - (ca - cb))])
did = pd.DataFrame(rows, columns=["Event", "Cell_ID", "Neighbor_Cell_ID", "Same_Site", "Technology",
                                  "Duration_h", "Duration_Group", "Block", "Season", *KPIS])
did["Neighbor_Type"] = np.where(did.Same_Site, "Same site", "Other site")
did["Duration_Group"] = pd.Categorical(did.Duration_Group, ["<1 h", "1-2 h", "2-3 h", "3-4 h", ">4 h"], ordered=True)
print(f"Pairs with complete windows: {len(did):,} (from {did.Event.nunique():,} outages)")
did.to_parquet(OUT_DIR / "step5_did_pairs.parquet", index=False)

In [ ]:
# ============================================================
# Step 5C — breakdown by group with sign tests
# ============================================================
SHOW = ["Load", "Connected_Users", "Traffic", "Latency"]

def summarise(group_col):
    out_rows = []
    for level, g in did.groupby(group_col, observed=True):
        r = {"Group": group_col, "Level": level, "Pairs": len(g)}
        for k in SHOW:
            nz = g[k][g[k] != 0]
            up = int((nz > 0).sum())
            r[f"{k} % up"] = 100 * up / len(nz) if len(nz) else np.nan
            r[f"{k} p"] = binomtest(up, len(nz)).pvalue if len(nz) else np.nan
        out_rows.append(r)
    return out_rows

table = pd.DataFrame(summarise("Neighbor_Type") + summarise("Duration_Group") +
                     summarise("Technology") + summarise("Block"))
pd.set_option("display.float_format", lambda v: f"{v:.3g}")
display(table)
table.to_csv(OUT_DIR / "step5_rq2_by_group.csv", index=False)
print("% up = share of pairs where the neighbor KPI rose more than in the control week (50% = no effect).")
print("p = two-sided sign test. With many groups, use p < 0.05 / number of tests (Bonferroni).")

rho, p_rho = spearmanr(did.Duration_h, did.Load)
print(f"\nDose-response: Spearman correlation between outage duration and neighbor Load DiD = {rho:.3f} (p = {p_rho:.2g})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, col, color in [(axes[0], "Neighbor_Type", "#2a78d6"), (axes[1], "Duration_Group", "#eb6834")]:
    sub = table[table.Group == col]
    ax.bar(sub.Level.astype(str), sub["Load % up"], color=color)
    ax.axhline(50, color="gray", ls="--")
    for x, v in enumerate(sub["Load % up"]):
        ax.text(x, v + 0.5, f"{v:.0f}%", ha="center")
    ax.set_title(f"Neighbor Load up vs control — by {col.replace('_', ' ').lower()}", loc="left", fontsize=10)
axes[0].set_ylabel("% of outage-neighbor pairs with higher Load")
axes[0].set_ylim(30, max(75, table["Load % up"].max() + 5))
plt.tight_layout(); plt.savefig(OUT_DIR / "step5_rq2_groups.png", dpi=200); plt.show()

## Step 6 — Summary for the report / สรุปไปใส่รีพอร์ต

**What it does / ทำอะไร:** prints the key numbers in sentences you can paste into the report, and lists every saved file.
พิมพ์ตัวเลขสำคัญเป็นประโยคพร้อมใส่รีพอร์ต และแสดงรายชื่อไฟล์ที่บันทึกไว้ทั้งหมด

**Send me the output of this cell (or the files in `outputs/`) and I will write the report section and slides from it.**
**ส่งผลของ cell นี้ (หรือไฟล์ในโฟลเดอร์ `outputs/`) มาให้ จะเขียนส่วนรีพอร์ตและสไลด์ให้จากผลจริง**

In [ ]:
# ============================================================
# Step 6 — key numbers
# ============================================================
def irr_line(name):
    if name not in irr.index:
        return f"- {name}: not in model"
    r = irr.loc[name]
    return f"- {name}: IRR {r.IRR:.2f} (95% CI {r.CI_low:.2f}-{r.CI_high:.2f}), p = {r.p_value:.2g}"

print("RQ3 (outage risk, Poisson model)")
for name in irr.index:
    print(irr_line(name))
m, b = res.iloc[0, 0], res.iloc[1, 0]
print(f"- Forecast check: model MAE {m:.1f} vs flat average {b:.1f} outages per region-month "
      f"({abs(1 - m / b) * 100:.0f}% {'lower = better' if m < b else 'higher = worse'})")

print("\nRQ2 extension (neighbor impact, DiD)")
print(f"- {len(did):,} outage-neighbor pairs from {did.Event.nunique():,} outages")
for _, r in table[table.Group.isin(["Neighbor_Type", "Duration_Group"])].iterrows():
    print(f"- {r.Group} = {r.Level}: Load up in {r['Load % up']:.1f}% of pairs (p = {r['Load p']:.2g}, n = {r.Pairs:,})")
print(f"- Duration vs Load DiD: Spearman rho = {rho:.3f} (p = {p_rho:.2g})")

print("\nSaved files:")
for f in sorted(OUT_DIR.glob("*")):
    print("  ", f)